# Week 5 studio — REFERENCE SOLUTION (instructor-only)

**Task brief:** [`README.md`](README.md) · **Lesson plan:** [`../../weeks/week-05.md`](../../weeks/week-05.md) · **Given engine:** [`dh_pki.py`](dh_pki.py)

Teaching walkthrough of the week-5 studio (Diffie–Hellman, MITM, cert chains, the
trust anchor). It **imports the reference answers from [`solution.py`](solution.py)**
— never re-pasting them — so what runs here is exactly what `test_dh_pki.py` grades.
Correctness is verified separately: `python3 studios/_verify_solutions.py week-05`.

Two moments below are *guarantee failures* staged live: an unauthenticated-DH MITM,
and a forged chain accepted the instant the trust store is poisoned. Watching a
guarantee fail once is how you learn it was conditional all along.

> Do not distribute. Excluded from students via `studios/.gitignore`.

In [ ]:
# --- bootstrap: week folder (for solution + engine) + repo root (for seclab) ---
import sys, pathlib
here = pathlib.Path.cwd()
week = here if (here / "solution.py").exists() else here / "studios" / "week-05"
root = week.parent.parent
for p in (str(week), str(root)):
    if p not in sys.path:
        sys.path.insert(0, p)

import inspect
import solution
import dh_pki
from dh_pki import DH_P, DH_G, validate

FX = dh_pki.load_fixtures()
# Fixed private exponents so the run is deterministic (any ints work for DH).
A_PRIV = 0x1a2b3c4d5e6f7788
B_PRIV = 0x99aabbccddeeff00
M_PRIV = 0xdeadbeefcafef00d      # Mallory
print("loaded RFC 2409 group-2 (1024-bit) DH group and the seed-7 PKI fixtures")

## Task 1a — honest DH: a shared secret over a hostile wire

`dh_public` sends `g^private mod p`; `dh_shared` raises the *other* side's public
value to *my* private exponent. Run against each other, Alice and Bob land on the
same `g^(ab) mod p`. DH's guarantee (axis 2): **secret vs. a passive eavesdropper**
who saw only the two public values (discrete-log assumption). Mirrors
`test_dh_direct_agreement_is_secret_and_shared`.

In [ ]:
print(inspect.getsource(solution.dh_public))
print(inspect.getsource(solution.dh_shared))

A = solution.dh_public(A_PRIV)
B = solution.dh_public(B_PRIV)
ka = solution.dh_shared(B, A_PRIV)   # Alice: Bob's public ^ Alice's private
kb = solution.dh_shared(A, B_PRIV)   # Bob:   Alice's public ^ Bob's private
assert ka == kb, "honest DH: Alice and Bob did not derive the same secret"
assert ka == pow(DH_G, A_PRIV * B_PRIV, DH_P), "shared secret != g^(ab) mod p"
print("honest DH: Alice and Bob agree on g^(ab) mod p")

## Task 1b — the guarantee fails: unauthenticated DH is MITM'd

DH gives secrecy on *each leg* and **zero authentication of the endpoints**. Mallory
injects her own public value toward BOTH sides: she now shares a key with each of
Alice and Bob, and **Alice and Bob share nothing**. The math never broke — the
missing property was never there. This stages `test_mitm_breaks_unauthenticated_dh`.

| Property | Bucket | Under MITM |
|---|---|---|
| Secrecy on each leg (vs. passive eavesdropper) | **GUARANTEE (conditional)** | still holds — each leg's key is secret |
| Endpoint authentication | **absent** | Mallory impersonates both ends; the guarantee is void |

In [ ]:
print(inspect.getsource(solution.mitm_keys))

mk = solution.mitm_keys(A_PRIV, B_PRIV, M_PRIV)
assert mk["alice"] == mk["mallory_alice"], "Mallory does not share Alice's key"
assert mk["bob"] == mk["mallory_bob"], "Mallory does not share Bob's key"
assert mk["alice_equals_bob"] is False   # the guarantee failing, live
assert mk["alice"] != mk["bob"]
print("GUARANTEE-FAIL: Mallory holds a key with EACH side; Alice<->Bob share nothing.")
print("LESSON: secrecy without authentication is a private conversation with an impostor.")

## Task 2 — cert-chain validation & two forgeries (given `validate`)

`validate` walks leaf → intermediate → root and requires the root to be a trusted
anchor. The legitimate chain validates; both forgeries are rejected at the **same**
check — they do not terminate in a trusted root. Mallory can sign anything; she
cannot make your browser *trust* her signing key. Mirrors
`test_legitimate_chain_validates`, `test_self_signed_forgery_rejected_at_anchor`,
`test_rogue_ca_chain_rejected_at_anchor`.

In [ ]:
ok, reason = validate(FX["chain"], FX["trust_store"])
assert ok, f"legitimate chain rejected: {reason}"
print("legit chain :", ok, "-", reason)

ok, reason = validate([FX["self_signed_forgery"]], FX["trust_store"])
assert not ok and "trusted root" in reason
print("self-signed :", ok, "-", reason)

chain = [FX["rogue_leaf"], FX["rogue_ca"]]
ok, reason = validate(chain, FX["trust_store"])
assert not ok and "trusted root" in reason
print("rogue-CA    :", ok, "-", reason, "  (SAME check as the self-signed forgery)")

## Task 3 — the attack moment: poison the trust store

The signatures never break — the **trust anchor** does. `poison_trust_store` returns
a NEW store that also trusts the rogue root. The same rogue chain that just failed
now validates. This is the DigiNotar (2011) failure mode in miniature. Stages
`test_trust_store_poisoning_accepts_forgery`.

- Clean store → rogue chain **REJECTED** (RAISES COST: Mallory must get her root trusted).
- Rogue root installed → rogue chain **ACCEPTED** (the guarantee — key bound to name —
  is only as strong as the trust store).

In [ ]:
print(inspect.getsource(solution.poison_trust_store))

chain = [FX["rogue_leaf"], FX["rogue_ca"]]
ok_before, _ = validate(chain, FX["trust_store"])
assert not ok_before, "rogue chain must be rejected under the clean store"

poisoned = solution.poison_trust_store(FX["trust_store"], FX["rogue_ca"])
assert FX["rogue_ca"]["pub"] not in FX["trust_store"], "must not mutate caller's store"
ok_after, reason = validate(chain, poisoned)
assert ok_after, f"poisoned store still rejects: {reason}"
print("before (clean store)   :", ok_before)
print("after  (rogue trusted) :", ok_after, "-", reason)
print("ATTACK MOMENT: rejected clean, ACCEPTED once the rogue root is trusted.")
print("LESSON: PKI's guarantee is only as strong as the trust store; CT is the DETECTIVE control.")

## Control Scorecard (Task 4)

| Mechanism | Guarantee (axis 2) | Its condition / failure |
|---|---|---|
| Diffie–Hellman | shared secret vs. a **passive** eavesdropper | **no authentication** — an active MITM defeats it (shown above) |
| Certificate chain | binds a key to a name | only as trustworthy as the **root trust store** (poisoning shown above) |
| TLS 1.3 | confidential, authenticated channel | = authenticated **DH** + a **cert chain** + **AEAD**; every underlying condition (nonce, key, trust) must hold |

**Name the condition, find the attack.** Secrecy and authentication are different
properties; a channel needs both.